In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import ce # type: ignore
import linear # type: ignore
import softmax # type: ignore
from common import repeat, test_checkgrad_2, test_compare_2, test_model_sgd_2 # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_Softmax_CE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
    
    def __init__(self, in_features: int, out_features: int, w: tr.Tensor = None, b: tr.Tensor = None):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.loss_ = nn.CrossEntropyLoss(reduction='mean')

        if w is not None:
            with tr.no_grad():
                self.linear.weight.copy_(w)

        if b is not None:
            with tr.no_grad():
                self.linear.bias.copy_(b)

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        return self.linear(x)

    def loss(self, p, t):
        return self.loss_(p, t)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1).float()

In [ ]:

class Per_Lin_Softmax_CE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features, out_features, w: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        self.linear = linear.Linear(in_features, out_features, w, b)
        self.softmax = softmax.Softmax()
        self.loss_ = ce.CE(reduction='mean')

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        return self.linear(x)
    
    def loss(self, p, t):
        return self.loss_(self.softmax(p), t)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1).float()       

$$ \text{Definition} $$

$$ z = xw + b $$

$$ p_{i}(z) = \frac{e^{z_i}}{\sum_{k} e^{z_k}} $$

$$ L(p, t) = -\sum_i t_i \, \ln(p_i)  $$

$$ F(x, w, b, t) = L(x, w, b, t) $$

$$ \sum_i p_i = 1, \quad \sum_i t_i = 1 $$

$$ \text{Derivatives} $$

$$ \frac{dz}{dw} = x $$

$$ \frac{dz}{db} = 1 $$

$$ \frac{dp_i}{dz_j} = p_i (\delta_{ij} - p_j) $$

$$ \frac{dp}{dz} = \text{diag}(p) - p p^\top $$

$$ \frac{dL}{dp_i} = -t_i \frac{1}{p_i} $$

$$ \frac{dL}{dp} = -t \oslash p  $$

$$ \frac{dp_i}{dz_j} = p_i (\delta_{ij} - p_j) $$

$$ \frac{dp}{dz} = \text{diag}(p) - p p^\top $$

$$ \frac{dL}{dz} \overset{*}{=} p - t $$

$$ \\[2em] $$

$$
*) \; \frac{dL}{dz_j} = 
\sum_i \frac{dL}{dp_i} \frac{dp_i}{dz_j} = \\
\sum_i \Big( -t_i \frac{1}{p_i} \Big) \Big( p_i (\delta_{ij} - p_j) \Big) = \sum_i -t_i \delta_{ij} + \sum_i t_i p_j = \\
-t_j + p_j
$$

$$ \text{Frobenius equality} $$

$$ \left \langle \frac{dF}{dw}, dw \right \rangle + \left \langle \frac{dF}{db}, db \right \rangle = \left \langle \frac{dF}{dL}, dL \right \rangle $$

$$ \\[2em] $$

$$ \left \langle \frac{dF}{dL}, dL \right \rangle = \frac{dF}{dL} \left \langle \frac{dL}{dz}, dz \right \rangle =  $$

$$ 
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{dz}{dw} dw + \frac{dz}{db} db \right \rangle = 
$$

$$ 
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{dz}{dw} dw \right \rangle + 
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{dz}{db} db \right \rangle = 
$$

$$ 
\frac{dF}{dL} \left \langle \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dL}{dz}, dw \right \rangle + 
\frac{dF}{dL} \left \langle \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dL}{dz}, db \right \rangle = 
$$

$$ 
\left \langle \frac{dF}{dL} \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dL}{dz}, dw \right \rangle + 
\left \langle \frac{dF}{dL} \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dL}{dz}, db \right \rangle \implies 
$$

$$ 
\begin{dcases}
\frac{dF}{dw} = \frac{dF}{dL} \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dL}{dz} \\
\frac{dF}{db} = \frac{dF}{dL} \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dL}{dz}
\end{dcases}
$$


In [ ]:
class Per_Lin_Softmax_CE_Gradient_Function(autograd.Function):
    @staticmethod
    def __model(x, w, b):
        return linear.linear(x, w, b)
    
    @staticmethod
    def __loss(z, t):
        return ce.ce(softmax.softmax(z), t)
    
    @staticmethod
    def predict(x, w, b):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.__model(x, w, b).argmax(dim=1).float()

    @staticmethod
    def forward(ctx, x, w, b, t):
        z = linear.linear(x, w, b)
        p = softmax.softmax(z)
        L = ce.ce(p, t)
        
        ctx.save_for_backward(x, p, t)

        return L
    
    @staticmethod
    def backward(ctx, dF_dL):
        x, p, t = ctx.saved_tensors
        S = x.shape[0]

        dz_dw = x
        dL_dz = (1/S) * (p - t)
        dF_dw = dF_dL * (dz_dw.t() @ dL_dz)
        dF_db = dF_dL * dL_dz.sum(dim=0)

        return (None, dF_dw, dF_db, None)

    

class Per_Lin_Softmax_CE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `t`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, w: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `weight` has shape `(out_features, in_features)` to be `nn.Linear` compatible
        if w is None:
            self.weight = nn.Parameter(0.1 * tr.randn(out_features, in_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(w.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(0.1 * tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_Softmax_CE_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.__model(x, self.weight.t(), self.bias)
    
    def loss(self, p, t):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.__loss(p, t)

    def predict(self, x):
        with tr.no_grad():
            return Per_Lin_Softmax_CE_Gradient_Function.predict(x, self.weight.t(), self.bias)
        
    def forward(self, x, t):
        return Per_Lin_Softmax_CE_Gradient_Function.apply(x, self.weight.t(), self.bias, t)

In [ ]:
if __name__ == "__main__":
    def onehot(z: tr.Tensor) -> tr.Tensor:
        S, C = z.shape
        p = tr.softmax(z, dim=1)
        idx = p.argmax(dim=1)
        t = tr.zeros_like(p)
        t[tr.arange(S), idx] = 1.0

        return t

    test_checkgrad_2(Per_Lin_Softmax_CE_Gradient_Function, 2, 2, 2, prec=0.01, fY=onehot)
    test_checkgrad_2(Per_Lin_Softmax_CE_Gradient_Function, 3, 3, 3, prec=0.01, fY=onehot)
    test_checkgrad_2(Per_Lin_Softmax_CE_Gradient_Function, 5, 5, 5, prec=0.01, fY=onehot)

    test_compare_2(Per_Lin_Softmax_CE_Autograd, Per_Lin_Softmax_CE_Backward, 1, 1, 1, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Autograd, Per_Lin_Softmax_CE_Backward, 2, 3, 4, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Autograd, Per_Lin_Softmax_CE_Backward, 10, 20, 30, fY=onehot)

    test_compare_2(Per_Lin_Softmax_CE_Backward, Per_Lin_Softmax_CE_Gradient, 1, 1, 1, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Backward, Per_Lin_Softmax_CE_Gradient, 2, 3, 4, fY=onehot)
    test_compare_2(Per_Lin_Softmax_CE_Backward, Per_Lin_Softmax_CE_Gradient, 10, 20, 30, fY=onehot)